# Vespa Redeploy — Fast BGE-M3 Hybrid

Key changes vs original deployment:
1. **HNSW index** on `dense_rep` → ANN is fast lookup, not brute-force scan
2. **Two-phase ranking** → first phase uses only `dense + lexical` (cheap) on all candidates, ColBERT `max_sim` runs only on top-10 in second phase
3. **`colbert_rep` moved to `index` + `attribute`** so Vespa can lazily fetch it only for second-phase docs

In [1]:
import pandas as pd
import json
import torch
from tqdm import tqdm
from FlagEmbedding import BGEM3FlagModel
from vespa.package import (
    Schema, Document, Field, FieldSet,
    RankProfile, Function,
    FirstPhaseRanking, SecondPhaseRanking,
    ApplicationPackage,
)
from vespa.deployment import VespaCloud
from vespa.application import Vespa

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

CORPUS_CSV_PATH  = "company_ne.csv"
QUERIES_JSON_PATH = "eval_queries_ne.json"
TENANT_NAME      = "anupstenant"

/home/an00b/Anup2026/Software/omnidata-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


/home/an00b/Anup2026/Software/omnidata-rag/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(device == "cuda"))

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 13266.12it/s]


In [3]:
df = pd.read_csv(CORPUS_CSV_PATH).dropna(subset=['text', 'id', 'section'])
corpus_texts = (df['section'] + " \n " + df['text']).tolist()
print(f"Loaded {len(corpus_texts)} documents.")

Loaded 165 documents.


## 1 — Schema

Changes from original:
- `dense_rep`: added `"index"` to indexing list + `index="hnsw"` → enables fast ANN
- `m3hybrid` rank profile: **first phase** = `dense + lexical` only (fast), **second phase** = full `max_sim` on `rerank-count: 10` candidates only

In [4]:
nepali_schema = Schema(
    name="nepali_docs",
    document=Document(
        fields=[
            Field(name="id",   type="string", indexing=["summary", "attribute"]),
            Field(name="text", type="string", indexing=["summary", "index"], index="enable-bm25"),
            Field(
                name="lexical_rep",
                type="tensor<bfloat16>(t{})",
                indexing=["summary", "attribute"],
            ),
            Field(
                name="dense_rep",
                type="tensor<bfloat16>(x[1024])",
                indexing=["summary", "attribute", "index"],  # <-- added "index"
                attribute=["distance-metric: angular"],
                index="hnsw",                                 # <-- HNSW index
            ),
            Field(
                name="colbert_rep",
                type="tensor<bfloat16>(t{}, x[1024])",
                indexing=["summary", "attribute"],
            ),
        ]
    ),
    fieldsets=[FieldSet(name="default", fields=["text"])],
)

m3hybrid = RankProfile(
    name="m3hybrid",
    inputs=[
        ("query(q_dense)",       "tensor<bfloat16>(x[1024])"),
        ("query(q_lexical)",     "tensor<bfloat16>(t{})"),
        ("query(q_colbert)",     "tensor<bfloat16>(qt{}, x[1024])"),
        ("query(q_len_colbert)", "float"),
    ],
    functions=[
        Function(
            name="dense",
            expression="cosine_similarity(query(q_dense), attribute(dense_rep), x)",
        ),
        Function(
            name="lexical",
            expression="sum(query(q_lexical) * attribute(lexical_rep))",
        ),
        Function(
            name="max_sim",
            expression="sum(reduce(sum(query(q_colbert) * attribute(colbert_rep), x), max, t), qt) / query(q_len_colbert)",
        ),
    ],
    # Phase 1: cheap — runs on ALL candidates
    first_phase=FirstPhaseRanking(
        expression="0.6*dense + 0.4*lexical"
    ),
    # Phase 2: expensive ColBERT — runs on top-10 only
    second_phase=SecondPhaseRanking(
        expression="0.4*dense + 0.2*lexical + 0.4*max_sim",
        rerank_count=10,
    ),
)

nepali_schema.add_rank_profile(m3hybrid)
vespa_app_package = ApplicationPackage(name="nepalim3", schema=[nepali_schema])
print("Schema defined.")

Schema defined.


## 2 — Deploy

In [6]:
vespa_cloud = VespaCloud(
    tenant=TENANT_NAME,
    application="nepali-m3-eval2",
    application_package=vespa_app_package,
)
app = vespa_cloud.deploy()
print("Deployed.")

Setting application...
Running: vespa config set application anupstenant.nepali-m3-eval2.default
Setting target cloud...
Running: vespa config set target cloud

No api-key found for control plane access. Using access token.
Checking for access token in auth.json...
Successfully obtained access token for control plane access.
Certificate and key not found in /home/an00b/Anup2026/Software/omnidata-rag/test/.vespa or /home/an00b/.vespa/anupstenant.nepali-m3-eval2.default: Creating new cert/key pair with vespa CLI.
Generating certificate and key...
Running: vespa auth cert -N
Success: Certificate written to '/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-public-cert.pem'
Success: Private key written to '/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-private-key.pem'

Deployment started in run 1 of dev-aws-us-east-1c for anupstenant.nepali-m3-eval2. This may take a few minutes the first time.
INFO    [06:23:33]  Deploying platform version 8.692.16 and a

## 3 — Re-feed

HNSW index is built during feed, so all docs must be re-fed after the schema change.

In [7]:
print("Encoding corpus...")
embeddings = model.encode(
    corpus_texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
    batch_size=32,
)
print("Encoding done.")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Encoding corpus...
Encoding done.


In [8]:
print("Feeding documents...")
errors = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    colbert_vecs = embeddings['colbert_vecs'][i]  # shape (num_tokens, 1024)
    fields = {
        "id":          str(row['id']),
        "text":        corpus_texts[i],
        "lexical_rep": {str(k): float(v) for k, v in embeddings['lexical_weights'][i].items()},
        "dense_rep":   embeddings['dense_vecs'][i].tolist(),
        "colbert_rep": {str(j): colbert_vecs[j].tolist() for j in range(colbert_vecs.shape[0])},
    }
    resp = app.feed_data_point(
        schema="nepali_docs",
        data_id=str(row['id']),
        fields=fields,
    )
    if resp.status_code != 200:
        errors.append((row['id'], resp.status_code))

print(f"Feed complete. Errors: {len(errors)}")
if errors:
    print(errors)

Feeding documents...


 99%|█████████▉| 163/165 [30:23<00:22, 11.18s/it]


IndexError: list index out of range

## 4 — Smoke test

In [2]:
VESPA_URL   = "https://cae4a7e4.a40e1008.z.vespa-app.cloud/"
VESPA_CERT  = "/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-public-cert.pem"
VESPA_KEY   = "/home/an00b/.vespa/anupstenant.nepali-m3-eval2.default/data-plane-private-key.pem"
from vespa.application import Vespa

app = Vespa(
    url=VESPA_URL,
    cert=VESPA_CERT,
    key=VESPA_KEY
)

In [4]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3")

/home/an00b/Anup2026/Software/omnidata-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/an00b/Anup2026/Software/omnidata-rag/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 67541.13it/s]


In [5]:
query = "कर्मचारी बोनस"

q = model.encode([query], return_dense=True, return_sparse=True, return_colbert_vecs=True)

colbert_vecs  = q["colbert_vecs"][0]
q_colbert     = {str(i): vec.tolist() for i, vec in enumerate(colbert_vecs)}
q_len_colbert = float(len(colbert_vecs))

response = app.query(
    yql="""
        select id from nepali_docs where
        userQuery() or
        ({targetHits:10}nearestNeighbor(dense_rep, q_dense));
    """,
    ranking="m3hybrid",
    hits=5,
    body={
        "input.query(q_dense)":       q["dense_vecs"][0].tolist(),
        "input.query(q_lexical)":     {str(k): float(v) for k, v in q["lexical_weights"][0].items()},
        "input.query(q_colbert)":     q_colbert,
        "input.query(q_len_colbert)": q_len_colbert,
        "timeout": "30s", 
    },
    query=query,
)

print(f"Status: {response.status_code}")
for hit in response.hits:
    print(hit['fields']['id'], '  relevance:', hit['relevance'])

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Status: 200
company_21_006   relevance: 0.4607355369574657
company_06_006   relevance: 0.4399986094933179
company_04_038   relevance: 0.4160407331478957
company_06_016   relevance: 0.4042082723871168
company_05_015   relevance: 0.39930898395246694


In [ ]:
# # Delete all existing docs first
# app.delete_all_docs(schema="nepali_docs", namespace="nepali_docs")
# print("Deleted all docs.")

TypeError: Vespa.delete_all_docs() missing 1 required positional argument: 'content_cluster_name'

## (Optional) Reconnect without redeploying
Use this cell in future sessions to skip deploy/feed and go straight to querying.

In [ ]:
# app = Vespa(
#     url="https://ca9355dd.d22d6d0a.z.vespa-app.cloud/",
#     cert="/home/an00b/.vespa/anupstenant.nepali-m3-eval.default/data-plane-public-cert.pem",
#     key="/home/an00b/.vespa/anupstenant.nepali-m3-eval.default/data-plane-private-key.pem",
# )

In [12]:
response = app.query(
    yql="""
        select id from nepali_docs where
        userQuery() or
        ({targetHits:10}nearestNeighbor(dense_rep, q_dense));
    """,
    ranking="m3hybrid",
    hits=5,
    body={
        "input.query(q_dense)":       q["dense_vecs"][0].tolist(),
        "input.query(q_lexical)":     {str(k): float(v) for k, v in q["lexical_weights"][0].items()},
        "input.query(q_colbert)":     q_colbert,
        "input.query(q_len_colbert)": q_len_colbert,
        "timeout": "30s",   # <-- add this
    },
    query=query,
)